<a href="https://colab.research.google.com/github/ShaneAdamczyk/Resume/blob/main/BAN482_Data_Exploration_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Plotly Tutorial

A hands-on guide to creating interactive visualizations, 3D charts, maps, subplots, and animations with Python's Plotly library.

## Introduction to Plotly in Python

Plotly is a powerful Python graphing library that makes it easy to create beautiful, interactive, and publication-quality visualizations. It offers two primary APIs:

1.  **Plotly Express (`px`)**: A high-level, concise API that's perfect for rapid data exploration and creating common chart types with minimal code. It's the recommended starting point.
2.  **`graph_objects` (`go`)**: A lower-level API that provides maximum control over every aspect of your plot. You'll use this for more complex or highly customized charts, like subplots or adding custom controls.

Plotly charts are inherently interactive—you can zoom, pan, and hover over data points to get more information right out of the box.



---

## Getting Started

First, you'll need to install the necessary packages. You can install Plotly and the datasets we'll use with pip.

In [1]:
!pip install plotly pandas

---

## High-Level Plotly Express

Plotly Express (`px`) is the fastest way to create a figure. Let's make a scatter plot using the built-in Iris dataset.

The code is incredibly simple. We import the library, load a dataset, and call the `px.scatter` function, specifying the data and mapping columns to the x-axis, y-axis, and color.

In [2]:
import plotly.express as px
import pandas as pd

# Load the Iris dataset
df_iris = px.data.iris()

# Create a scatter plot
fig = px.scatter(df_iris,
                 x="sepal_width",
                 y="sepal_length",
                 color="species", # Assign color based on a categorical feature
                 size='petal_length', # Assign size of the point based on a numeric value
                # hover_data=['petal_width'], # Add any other data you want to show upon hovering
                 title="Iris Dataset: Sepal Length vs. Width")

# Show the figure
fig.show()

---

### Saving Plotly Figures as HTML

To save an interactive Plotly figure as a standalone HTML file, you can use the `.write_html()` method of the figure object. This is useful for embedding figures in websites, presentations, or sharing them with others without requiring a Python environment.

In [ ]:
# Save the last generated figure (assuming 'fig' is still the active figure object)
fig.write_html("my_plotly_figure.html")

print("Figure saved as 'my_plotly_figure.html'. You can download it from the file browser on the left sidebar.")

## Lower-Level `graph_objects`

For fine-grained control, we use `graph_objects` (`go`). Let's recreate the same scatter plot. You'll notice the syntax is more verbose but also more explicit.

In [ ]:
import plotly.graph_objects as go
import pandas as pd

# We use the same Iris dataset, loaded from a URL for this example
df_iris = pd.read_csv('https://raw.githubusercontent.com/plotly/datasets/master/iris-data.csv')

# Create an empty figure
fig = go.Figure()

# Add a trace (a "trace" is a collection of data and its visual representation)
# We will add one trace for each species to control their color
for species in df_iris['class'].unique():
    df_species = df_iris[df_iris['class'] == species]
    fig.add_trace(go.Scatter(
        x=df_species['sepal width'],
        y=df_species['sepal length'],
        mode='markers', # We only want markers, not lines
        name=species, # This will appear in the legend
        marker=dict(size=10)
    ))

# Update the layout to add titles and labels
fig.update_layout(
    title="Iris Dataset: Sepal Length vs. Width (using graph_objects)",
    xaxis_title="Sepal Width",
    yaxis_title="Sepal Length"
)

fig.show()

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3D Charts

Creating 3D charts is just as easy with Plotly Express. You simply use a 3D-specific function like `px.scatter_3d`.

In [ ]:
import plotly.express as px

# Using the same Iris dataset
df_iris = px.data.iris()

# Create a 3D scatter plot
fig = px.scatter_3d(df_iris,
                    x='sepal_length',
                    y='sepal_width',
                    z='petal_width',
                    color='species',
                    title="3D Scatter Plot of Iris Dataset")

fig.show()

---

## Maps

Plotly is excellent for creating geographical maps. Let's make a **choropleth map**, which shades countries based on a data variable (like GDP per capita).

In [ ]:
import plotly.express as px

# Load the Gapminder dataset, which contains country data over time
df_gapminder = px.data.gapminder()

# Filter data for a specific year, e.g., 2007
df_2007 = df_gapminder[df_gapminder['year']==2007]

# A more concise way of subsetting is shown below
# df_2007 = df_gapminder.query("year == 2007")

# Create the choropleth map
fig = px.choropleth(df_2007,
                    locations="iso_alpha", # Column with ISO 3166-1 alpha-3 country codes
                    color="gdpPercap", # Column to use for color scale
                    hover_name="country", # Column to display on hover
                    # title="Global GDP Per Capita in 2007",
                    color_continuous_scale=px.colors.sequential.Plasma # Color scale
                    )

fig.show()

---

## Subplots

To display multiple charts in a single figure, use the `make_subplots` utility. This is where `graph_objects` becomes essential, as Plotly Express can't directly create subplots.

Here, we'll create a figure with two plots side-by-side: a scatter plot and a bar chart.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px # import express to get the dataset easily

# Using the Gapminder dataset
df_gapminder = px.data.gapminder().query("continent == 'Oceania'")

# Use Method Chaining to make the code look elegant, easily understandable,less
# intimidating, and looks cool in front of python programmers
df_gapminder = (px
                .data
                .gapminder()
                .query("continent == 'Oceania'")
                )

# Initialize a figure with 1 row and 2 columns
fig = make_subplots(rows=1, cols=2, subplot_titles=("Life Expectancy vs. GDP", "Population"))

# --- First subplot (Scatter) ---
fig.add_trace(
    go.Scatter(x=df_gapminder['gdpPercap'],
               y=df_gapminder['lifeExp'],
               mode='markers'),
    row=1, col=1
)

# --- Second subplot (Bar) ---
fig.add_trace(
    go.Bar(x=df_gapminder['country'],
           y=df_gapminder['pop']),
    row=1, col=2
)

# Update overall layout
fig.update_layout(title_text="Oceania Dashboard (1952-2007)", showlegend=False)

fig.show()

---

## Animations

Animations can powerfully illustrate how data changes over a variable, like time.

Plotly Express makes this incredibly straightforward with the `animation_frame` argument.

Let's visualize how life expectancy and GDP have evolved for countries over the years.

In [ ]:
import plotly.express as px

df_gapminder = px.data.gapminder()

In [ ]:
df_gapminder.head()

In [ ]:

fig = px.scatter(df_gapminder,
                 x="gdpPercap",
                 y="lifeExp",
                 animation_frame="year", # Creates the animation slider based on this value.
                 animation_group="country", # Sets each trace to a unique country
                 size="pop", # size of the point marker will depend on the population
                 color="continent", # Color of the point marker will depend on the continent
                 hover_name="country", # name of country will be displayed when you hover over the marker
                 log_x=True, # Use a log scale for the x-axis for better visualization
                 size_max=55, # sets maximum size of the point markers
                 range_x=[100,100000], # Range for the x axis
                 range_y=[25,90], # Range for the y-axis. These settings help with consistency
                 title="The Evolution of Health and Wealth of Nations"
                 )

fig.show()

---

## Rich Interactivity with Buttons and Dropdowns

Finally, let's add custom controls to a plot using `graph_objects`. We'll create a plot with a dropdown menu that lets you switch between viewing linear and logarithmic axes.

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Create some sample data
x = np.arange(1, 101)
y_linear = 2 * x + 5
y_log = np.log(x)

fig = go.Figure()

# Add initial scatter trace
fig.add_trace(go.Scatter(x=x, y=y_linear, mode='lines', name='Linear'))

# Add buttons for dropdown menu
fig.update_layout(
    title="Plot with Dropdown Menu to Change Axis Type",
    xaxis_type="linear", # Initial axis type
    # Code below adds options
    updatemenus=[
        dict(
            buttons=list([
                dict(
                    args=[{"xaxis.type": "linear"}],
                    label="Linear Scale",
                    method="relayout"
                ),
                dict(
                    args=[{"xaxis.type": "log"}],
                    label="Log Scale",
                    method="relayout"
                )
            ]),
            # The code below defines settings for the dropdown bar
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.1,
            xanchor="left",
            y=1.1,
            yanchor="top"
        )
    ]
)

fig.show()

---

In [10]:
import pandas as pd

# Load the Spotify sample dataset using the provided file path
df_spotify = pd.read_csv('/spotify_sample.csv')

# Display the first 5 rows
display(df_spotify.head())

,artist_name,track_name,track_id,popularity,year,genre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature
0,Cabas,Amor De Mis Amores,3FTqjWmL21xi4HTTOq94EQ,38,2006,alt-rock,0.725,0.553,6,-6.319,0,0.0340,0.276000,0.000007,0.1850,0.729,90.009,206653,4
1,Ketil Bjørnstad,Første sang,0Pt7ESPgrdTdaxp2f29hX2,11,2023,swedish,0.277,0.164,9,-16.743,0,0.0373,0.878000,0.000181,0.3350,0.184,89.308,459733,4
2,Project 86,Evil (A Chorus Of Resistance),75Ub3ckaoTdzgH9Azeu8cY,38,2007,alt-rock,0.486,0.927,2,-4.845,0,0.0428,0.000003,0.014500,0.0952,0.377,135.540,183373,4
3,Ital Tek,Open Heart,5WEPna9GWi0NkqVLAkEKNN,18,2020,dubstep,0.411,0.442,1,-12.745,0,0.0270,0.485000,0.926000,0.1910,0.172,174.019,347610,3
4,I-Roy,Irie Right,6peHySxvmZaRF9YEwUsggq,18,2017,dancehall,0.748,0.660,10,-4.648,0,0.2710,0.125000,0.000000,0.0783,0.400,75.583,196179,4


In [11]:
# Get the number of rows in the DataFrame
num_rows = df_spotify.shape[0]
print(f"The dataset has {num_rows} rows.")

The dataset has 100000 rows.


In [12]:
import plotly.express as px

# Calculate the top 25 artists by track count
top_artists = df_spotify['artist_name'].value_counts().head(25).reset_index()
top_artists.columns = ['artist_name', 'track_count']

# Create a bar chart using Plotly Express
fig = px.bar(top_artists,
             x='artist_name',
             y='track_count',
             title='Top 25 Artists by Number of Tracks',
             labels={'artist_name': 'Artist Name', 'track_count': 'Number of Tracks'},
             color='track_count', # Color bars based on track count
             color_continuous_scale=px.colors.sequential.Plasma)

fig.update_layout(xaxis_tickangle=-45) # Rotate x-axis labels for better readability
fig.show()

### Scatter Plot: Popularity vs. Danceability

In [24]:
import plotly.express as px

# Sort the DataFrame by 'popularity' in descending order
df_popular = df_spotify.sort_values(by='popularity', ascending=False)

# Select the top N most popular songs to plot for better visualization clarity
# Let's take the top 500 for a good overview, adjust as needed.
# If there are too many points, the plot can become cluttered.
num_top_songs = 500
df_top_songs_for_plot = df_popular.head(num_top_songs)

fig = px.scatter(df_top_songs_for_plot,
                 x='danceability',
                 y='energy',
                 title=f'Energy vs. Danceability for Top {num_top_songs} Spotify Tracks',
                 labels={'danceability': 'Danceability', 'energy': 'Energy Score'},
                 hover_data=['track_name', 'artist_name', 'genre'])

fig.show()

In [34]:
import plotly.express as px

# Calculate the average energy for each year, excluding 2023
df_energy_by_year = df_spotify[df_spotify['year'] != 2023].groupby('year')['energy'].mean().reset_index()

# Create a line plot of average energy by year
fig = px.line(
    df_energy_by_year,
    x='year',
    y='energy',
    title='Average Energy of Spotify Tracks Over Years (Excluding 2023)',
    labels={'year': 'Year', 'energy': 'Average Energy'}
)

fig.show()

In [35]:
import plotly.express as px

# Calculate the average danceability for each year, excluding 2023
df_danceability_by_year = df_spotify[df_spotify['year'] != 2023].groupby('year')['danceability'].mean().reset_index()

# Create a line plot of average danceability by year
fig = px.line(
    df_danceability_by_year,
    x='year',
    y='danceability',
    title='Average Danceability of Spotify Tracks Over Years (Excluding 2023)',
    labels={'year': 'Year', 'danceability': 'Average Danceability'}
)

fig.show()

In [36]:
# Filter for 'rock-n-roll' genre
df_rock_n_roll = df_spotify[df_spotify['genre'].str.contains('rock-n-roll', case=False, na=False)]

# Filter out year 2023
df_rock_n_roll_filtered = df_rock_n_roll[df_rock_n_roll['year'] != 2023]

# Calculate the average popularity for 'rock-n-roll' for each year
df_rock_n_roll_popularity_by_year = df_rock_n_roll_filtered.groupby('year')['popularity'].mean().reset_index()

# Create a line plot of average popularity by year for 'rock-n-roll'
fig = px.line(
    df_rock_n_roll_popularity_by_year,
    x='year',
    y='popularity',
    title='Average Popularity of Rock-n-Roll Tracks Over Years (Excluding 2023)',
    labels={'year': 'Year', 'popularity': 'Average Popularity'}
)

fig.show()

In [32]:
min_year = df_spotify['year'].min()
max_year = df_spotify['year'].max()
print(f"The dataset covers tracks from {min_year} to {max_year}.")

The dataset covers tracks from 2000 to 2023.


In [37]:
# Filter for 'hip-hop' genre
df_hip_hop = df_spotify[df_spotify['genre'].str.contains('hip-hop', case=False, na=False)]

# Filter out year 2023
df_hip_hop_filtered = df_hip_hop[df_hip_hop['year'] != 2023]

# Calculate the average popularity for 'hip-hop' for each year
df_hip_hop_popularity_by_year = df_hip_hop_filtered.groupby('year')['popularity'].mean().reset_index()

# Create a line plot of average popularity by year for 'hip-hop'
fig = px.line(
    df_hip_hop_popularity_by_year,
    x='year',
    y='popularity',
    title='Average Popularity of Hip-Hop Tracks Over Years (Excluding 2023)',
    labels={'year': 'Year', 'popularity': 'Average Popularity'}
)

fig.show()

In [38]:
import plotly.express as px

# Filter out year 2023
df_spotify_filtered = df_spotify[df_spotify['year'] != 2023]

# Calculate the average popularity for each genre for each year
df_genre_popularity_by_year = df_spotify_filtered.groupby(['year', 'genre'])['popularity'].mean().reset_index()

# Create a line plot of average popularity by year for all genres
fig = px.line(
    df_genre_popularity_by_year,
    x='year',
    y='popularity',
    color='genre',
    title='Average Popularity of All Genres Over Years (Excluding 2023)',
    labels={'year': 'Year', 'popularity': 'Average Popularity', 'genre': 'Genre'}
)

fig.show()

In [43]:
# Calculate average popularity for each genre in 2000 and 2022 (excluding 2023)
df_2000_popularity = df_spotify[df_spotify['year'] == 2000].groupby('genre')['popularity'].mean().reset_index()
df_2000_popularity.rename(columns={'popularity': 'popularity_2000'}, inplace=True)

df_2022_popularity = df_spotify[df_spotify['year'] == 2022].groupby('genre')['popularity'].mean().reset_index()
df_2022_popularity.rename(columns={'popularity': 'popularity_2022'}, inplace=True)

# Merge the dataframes to compare popularity
df_popularity_comparison = pd.merge(
    df_2000_popularity,
    df_2022_popularity,
    on='genre',
    how='inner'
)

# Calculate the percentage decrease
df_popularity_comparison['popularity_decrease_pct'] = (
    (df_popularity_comparison['popularity_2000'] - df_popularity_comparison['popularity_2022']) /
    df_popularity_comparison['popularity_2000']
) * 100

# Filter for genres with a significant decrease (e.g., > 20%)
significant_decrease_threshold = 20 # Can be adjusted
genres_with_significant_decrease = df_popularity_comparison[
    df_popularity_comparison['popularity_decrease_pct'] > significant_decrease_threshold
]['genre'].tolist()

print(f"Genres with >{significant_decrease_threshold}% popularity decrease from 2000 to 2022: {genres_with_significant_decrease}")

Genres with >20% popularity decrease from 2000 to 2022: ['tango']


In [44]:
import plotly.express as px

# Filter the main popularity data for these genres and exclude 2023
df_decreasing_genres_popularity = df_genre_popularity_by_year[
    df_genre_popularity_by_year['genre'].isin(genres_with_significant_decrease)
]

# Create a line plot for these genres
fig = px.line(
    df_decreasing_genres_popularity,
    x='year',
    y='popularity',
    color='genre',
    title='Popularity Trends of Genres with >20% Decrease (2000-2022)',
    labels={'year': 'Year', 'popularity': 'Average Popularity', 'genre': 'Genre'},
    hover_name='genre'
)

fig.show()

In [41]:
# Determine the top 6 genres by popularity in 2022
df_2022 = df_spotify[df_spotify['year'] == 2022]
top_6_genres_2022 = df_2022.groupby('genre')['popularity'].mean().nlargest(6).index.tolist()

print(f"Top 6 genres by popularity in 2022: {top_6_genres_2022}")

Top 6 genres by popularity in 2022: ['pop', 'dance', 'house', 'hip-hop', 'rock', 'country']


In [42]:
import plotly.express as px

# Filter df_spotify for the top 6 genres from 2022 and exclude the year 2023
df_top_genres_filtered = df_spotify[
    (df_spotify['genre'].isin(top_6_genres_2022)) &
    (df_spotify['year'] != 2023)
]

# Calculate the average popularity for these top genres for each year
df_top_genre_popularity_by_year = df_top_genres_filtered.groupby(['year', 'genre'])['popularity'].mean().reset_index()

# Create a line plot of average popularity by year for the top 6 genres
fig = px.line(
    df_top_genre_popularity_by_year,
    x='year',
    y='popularity',
    color='genre',
    title='Average Popularity of Top 6 Genres (as of 2022) Over Years (Excluding 2023)',
    labels={'year': 'Year', 'popularity': 'Average Popularity', 'genre': 'Genre'},
    hover_name='genre'
)

fig.show()